# Data Exploration

Explore the **SOOP (Stroke Outcome Optimization Project)** dataset used for stroke detection with Graph Attention Networks on brain MRI.

## Two supported modes

This notebook auto-detects which dataset you have downloaded and adapts:

- **`bids` mode** (full reproduction): you've downloaded `ds004889` from OpenNeuro. Multi-modal (T1, FLAIR, ADC, TRACE) + 3-class labels (no lesion / acute / chronic).
- **`soop` mode** (FLAIR-only quickstart): you only have the SOOP normalized release. Single modality (FLAIR), single binary lesion mask, no acute/chronic split.

If neither dataset is available, the notebook prints a clear setup message and stops cleanly. See `README.md` Step 1 for download instructions.

In [ ]:
%matplotlib inline

import sys
sys.path.insert(0, "../src")

import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

from stroke_gat.config import load_config
from stroke_gat.data.service import DataService

In [ ]:
# Load project configuration
config = load_config("../configs/default.yaml")
print(f"Raw BIDS path:    {config.paths.raw_bids}")
print(f"SOOP path:        {config.paths.soop_normalized}")
print(f"Atlas file:       {config.paths.atlas_file}")
print(f"Atlas labels:     {config.paths.atlas_labels_file}")

service = DataService(config.paths)

In [ ]:
# Auto-detect mode: prefer BIDS (full multi-modal), fall back to SOOP normalized.
bids_subjects = service.discover_subjects_bids()
soop_subjects = service.discover_subjects_soop()

if bids_subjects:
    MODE = "bids"
    subjects = bids_subjects
    print(f"Mode: BIDS  ({len(subjects)} subjects, full multi-modal + 3-class)")
elif soop_subjects:
    MODE = "soop"
    subjects = soop_subjects
    print(f"Mode: SOOP-quickstart  ({len(subjects)} subjects, FLAIR-only + binary lesion)")
    print()
    print("  No raw BIDS data found at config.paths.raw_bids. Running in SOOP-quickstart")
    print("  mode (FLAIR-only). Acute/chronic labels and full multi-modal features")
    print("  require ds004889 from OpenNeuro --- see README Step 1.")
else:
    MODE = "none"
    subjects = []
    print("No data available.\n")
    print("Neither raw BIDS nor SOOP normalized data was found at the configured paths.")
    print("Edit configs/default.yaml (or copy to configs/local.yaml) and follow README")
    print("Steps 1-3 to download and configure the dataset.")

if subjects:
    print(f"\nFirst 10 IDs: {subjects[:10]}")

In [ ]:
# Load one subject's modalities. In BIDS mode all 4 modalities; in SOOP mode just FLAIR.
if MODE == "none":
    print("Skipping: no data available.")
else:
    subject_id = subjects[0]
    print(f"Loading subject: {subject_id}")

    if MODE == "bids":
        modalities = service.load_subject_modalities(subject_id)
    else:  # soop
        flair_img, _ = service.load_soop_normalized(subject_id)
        modalities = {"FLAIR": flair_img}

    print("\nLoaded modalities and shapes:")
    for name, img in modalities.items():
        data = img.get_fdata(dtype=np.float32)
        print(f"  {name:>8s}: shape={data.shape}, dtype={data.dtype}, "
              f"range=[{data.min():.3f}, {data.max():.3f}]")

In [ ]:
# Plot available modalities at the middle axial slice.
if MODE == "none":
    print("Skipping: no data available.")
else:
    available = [m for m in ["T1", "FLAIR", "ADC", "TRACE"] if m in modalities]
    ref_data = modalities[available[0]].get_fdata(dtype=np.float32)
    mid_slice = ref_data.shape[2] // 2

    fig, axes = plt.subplots(1, len(available), figsize=(5 * len(available), 5),
                              squeeze=False)
    axes = axes[0]
    for ax, name in zip(axes, available):
        vol = modalities[name].get_fdata(dtype=np.float32)
        ax.imshow(vol[:, :, mid_slice].T, cmap="gray", origin="lower")
        ax.set_title(name, fontsize=14, fontweight="bold")
        ax.axis("off")

    fig.suptitle(f"Subject {subject_id} - Axial Slice {mid_slice} ({MODE} mode)",
                 fontsize=16, fontweight="bold")
    plt.tight_layout()
    plt.show()

In [ ]:
# Load atlas (works in both modes; the atlas file path is independent of dataset mode).
if MODE == "none":
    print("Skipping: no data available.")
else:
    try:
        atlas_data, atlas_img = service.load_atlas()
        atlas_labels = service.load_atlas_labels()
        print(f"Atlas shape: {atlas_data.shape}")
        print(f"Number of unique regions: {len(np.unique(atlas_data)) - 1}")
        print(f"Sample labels: {dict(list(atlas_labels.items())[:5])}")

        ref_name = available[0]
        ref_vol = modalities[ref_name].get_fdata(dtype=np.float32)

        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        axes[0].imshow(ref_vol[:, :, mid_slice].T, cmap="gray", origin="lower")
        axes[0].set_title(ref_name)
        axes[0].axis("off")

        # Atlas overlay (only meaningful when subject and atlas share space).
        if atlas_data.shape == ref_vol.shape:
            axes[1].imshow(ref_vol[:, :, mid_slice].T, cmap="gray", origin="lower")
            atlas_slice = atlas_data[:, :, mid_slice].T.astype(float)
            atlas_masked = np.ma.masked_where(atlas_slice == 0, atlas_slice)
            axes[1].imshow(atlas_masked, cmap="nipy_spectral", alpha=0.4, origin="lower")
            axes[1].set_title(f"{ref_name} + ArterialAtlas136 Overlay")
        else:
            axes[1].text(0.5, 0.5,
                          f"Atlas shape {atlas_data.shape}\n!=\nsubject shape {ref_vol.shape}\n\n"
                          f"Atlas registration happens during preprocess_subject().\n"
                          f"Run scripts/preprocess_dataset.py to align them.",
                          ha="center", va="center", transform=axes[1].transAxes)
        axes[1].axis("off")
        plt.tight_layout()
        plt.show()
    except Exception as e:
        print(f"Atlas load failed: {e}")
        print("Set config.paths.atlas_file in configs/default.yaml. See README Step 2.")

In [ ]:
# Load lesion masks and visualize overlay.
# - BIDS mode: 3 masks (General, Acute, Chronic). 3-class colormap (red=acute, orange=chronic).
# - SOOP mode: 1 binary mask. 2-class colormap (red=lesion).
if MODE == "none":
    print("Skipping: no data available.")
elif MODE == "bids":
    masks = service.load_subject_masks(subject_id)
    metadata = DataService.get_subject_metadata(subject_id, masks)
    print(f"Stroke status: {metadata['stroke_status']}")
    print(f"Lesion level:  {metadata['lesion_level']}")
    print(f"Has acute: {metadata['has_acute']}, Has chronic: {metadata['has_chronic']}")

    flair_data = modalities["FLAIR"].get_fdata(dtype=np.float32) if "FLAIR" in modalities else ref_vol
    combined = np.zeros(flair_data.shape[:2], dtype=np.uint8)
    if masks["Acute"].ndim == 3:
        combined[masks["Acute"][:, :, mid_slice] > 0] = 1
    if masks["Chronic"].ndim == 3:
        combined[masks["Chronic"][:, :, mid_slice] > 0] = 2

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    axes[0].imshow(flair_data[:, :, mid_slice].T, cmap="gray", origin="lower")
    axes[0].set_title("FLAIR")
    axes[0].axis("off")

    lesion_cmap = ListedColormap(["black", "red", "orange"])
    axes[1].imshow(combined.T, cmap=lesion_cmap, vmin=0, vmax=2, origin="lower")
    axes[1].set_title("Lesion Mask (Red=Acute, Orange=Chronic)")
    axes[1].axis("off")

    axes[2].imshow(flair_data[:, :, mid_slice].T, cmap="gray", origin="lower")
    overlay = np.zeros((*combined.T.shape, 4))
    overlay[combined.T == 1] = [1, 0, 0, 0.5]
    overlay[combined.T == 2] = [1, 0.6, 0, 0.5]
    axes[2].imshow(overlay, origin="lower")
    axes[2].set_title("FLAIR + Lesion Overlay")
    axes[2].axis("off")

    fig.suptitle(f"Subject {subject_id} - Lesion Segmentation ({metadata['lesion_level']})",
                 fontsize=16, fontweight="bold")
    plt.tight_layout()
    plt.show()
else:  # soop
    flair_img, lesion_mask = service.load_soop_normalized(subject_id)
    flair_data = flair_img.get_fdata(dtype=np.float32)
    has_lesion = bool(lesion_mask.size and lesion_mask.any())
    print(f"Subject {subject_id}: lesion present = {has_lesion}")
    if not has_lesion:
        # Try to find a subject WITH a lesion for the visualization
        for sid in subjects[:50]:
            _, lm = service.load_soop_normalized(sid)
            if lm.size and lm.any():
                subject_id = sid
                flair_img, lesion_mask = service.load_soop_normalized(subject_id)
                flair_data = flair_img.get_fdata(dtype=np.float32)
                print(f"Using subject {subject_id} for visualization (has lesion).")
                break

    mid_slice = flair_data.shape[2] // 2
    if lesion_mask.shape != flair_data.shape:
        # Some SOOP files differ slightly in shape; pad/crop as needed for the demo slice.
        lesion_mask = np.zeros_like(flair_data, dtype=np.uint8)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].imshow(flair_data[:, :, mid_slice].T, cmap="gray", origin="lower")
    axes[0].set_title("FLAIR (warped to standard space)")
    axes[0].axis("off")

    axes[1].imshow(flair_data[:, :, mid_slice].T, cmap="gray", origin="lower")
    overlay = np.zeros((*lesion_mask[:, :, mid_slice].T.shape, 4))
    overlay[lesion_mask[:, :, mid_slice].T > 0] = [1, 0, 0, 0.5]
    axes[1].imshow(overlay, origin="lower")
    axes[1].set_title("FLAIR + Binary Lesion (single mask, no acute/chronic split)")
    axes[1].axis("off")

    fig.suptitle(f"Subject {subject_id} - SOOP-quickstart mode", fontsize=16, fontweight="bold")
    plt.tight_layout()
    plt.show()
    print("\nNote: SOOP-normalized lesion masks are SINGLE BINARY masks. The acute vs")
    print("chronic distinction the model uses for training lives only in the OpenNeuro")
    print("BIDS derivatives. See docs/DATA_LAYOUT.md, Section 4 for the labeling rule.")

## Summary

Key observations from the SOOP dataset:

1. **Multimodal complementarity** (BIDS mode only): Each MRI modality captures different tissue properties:
   - **T1:** Anatomical structure with good gray/white matter contrast
   - **FLAIR:** Highlights edema and chronic white matter changes
   - **ADC:** Quantifies water diffusion; restricted diffusion in acute infarct appears dark
   - **TRACE (DWI):** High signal in acute ischemic regions

2. **Class imbalance:** Lesion voxels constitute a small fraction of total brain volume, motivating weighted cross-entropy loss (Eq. 13).

3. **ArterialAtlas136:** Provides arterial territory context (32 regions) for anatomically-constrained SLIC supervoxel segmentation.

4. **Mode awareness:** Notebooks 02–05 require BIDS mode (full multi-modal data + 3-class labels). If you only have SOOP normalized, use this notebook for orientation, then download `ds004889` per README Step 1 before continuing.

Next: See `02_parcellation_and_slic_demo.ipynb` for supervoxel generation.